In [0]:
%sql
USE CATALOG taxi_data_silver;


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze_transformed_data;

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_transformed_data.taxi_data_transformed
USING DELTA
AS
WITH bronze AS 
  (SELECT 
    cast(date_format(br.tpep_pickup_datetime, 'yyyy-MM-dd') as date) AS pickup_date,
    cast(date_format(br.tpep_dropoff_datetime, 'yyyy-MM-dd') as date) AS dropoff_date,
    date_format(br.tpep_pickup_datetime, 'HH:mm:ss') AS pickup_time,
    date_format(br.tpep_dropoff_datetime, 'HH:mm:ss') AS dropoff_time,
    round((unix_timestamp(br.tpep_dropoff_datetime) - unix_timestamp(br.tpep_pickup_datetime))/60, 2) AS trip_dur_mins,
    br.trip_distance, 
    br.fare_amount, 
    br.pickup_zip, 
    br.dropoff_zip
  FROM taxi_data_bronze.data_ingestion.trips AS br
  )

SELECT 
  br2.*,
  CASE 
    WHEN br2.pickup_time BETWEEN '00:00:00' AND '05:59:59'  THEN 'late_night'
    WHEN br2.pickup_time BETWEEN '06:00:00' AND '10:59:59'  THEN 'morning_rush_hr'
    WHEN br2.pickup_time BETWEEN '11:00:00' AND '15:59:59'  THEN 'mid_day'
    WHEN br2.pickup_time BETWEEN '16:00:00' AND '19:59:59'  THEN 'evening_rush_hr'
    WHEN br2.pickup_time BETWEEN '20:00:00' AND '23:59:59'  THEN 'night'
    ELSE NULL END AS time_of_day,
  date_format(br2.pickup_date, 'EEEE') AS pickup_day_of_week
FROM bronze AS br2